In [7]:
import stable_retro as retro
import gymnasium as gym
from gymnasium import Env
from gymnasium.spaces import MultiBinary, Box
import numpy as np
import cv2
from matplotlib import pyplot as plt
import os
import optuna
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import SubprocVecEnv, VecFrameStack
from stable_baselines3 import PPO

In [8]:
class StreetFighter(Env):
    def __init__(self):
        super().__init__()
        self.observation_space = Box(low=0, high=255, shape=(84, 84, 1), dtype=np.uint8)
        
        self.action_space = MultiBinary(12)

        self.game = retro.make(game="StreetFighterIISpecialChampionEdition-Genesis-v0", use_restricted_actions=retro.Actions.FILTERED, render_mode=None)

    def step(self, action):
        obs, reward, terminated, truncated, info = self.game.step(action)
        obs = self.preprocess(obs)

        self.previous_frame = obs

        if info["health"] == 0 and info["enemy_health"] == 0:
            reward = 0
            self.enemy_health = 0
            self.player_health = 0
        else:
            dmg_dealt = self.enemy_health -info["enemy_health"]
            dmg_taken = self.player_health- info["health"]
            self.enemy_health = info["enemy_health"]
            self.player_health = info["health"]
            reward = dmg_dealt - dmg_taken
        

        return obs, reward, terminated, truncated, info


    def render(self, *args, **kwargs):
        self.game.render()

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        obs, info = self.game.reset(seed=seed, options=options)
        obs = self.preprocess(obs)
        self.previous_frame = obs

        
        info = self.game.data.lookup_all()
        self.player_health = info.get("health", 0)
        self.enemy_health = info.get("enemy_health", 0)
        return obs, info

    def preprocess(self, observation):
        gray = cv2.cvtColor(observation, cv2.COLOR_RGB2GRAY)

        resize = cv2.resize(gray, (84,84), interpolation=cv2.INTER_AREA)

        channels = np.reshape(resize, (84,84,1))
        return channels


    def close(self):
        self.game.close()

In [9]:
LOG_DIR = "./opt_logs/"
OPT_DIR = "./opt/"

In [ ]:
def make_env():
    return Monitor(StreetFighter(), LOG_DIR)

def make_vec_env():
    env = SubprocVecEnv([make_env for _ in range(4)]) # num of environments at ocne
    return VecFrameStack(env, n_stack=4, channels_order="last")

def opt_ppo(trial):
    return {
        "n_steps": trial.suggest_int("n_steps", 2048, 8192, step=64),
        "gamma": trial.suggest_float("gamma", 0.8, 0.9999, log=True),
        "learning_rate": trial.suggest_float("learning_rate", 1e-5, 1e-4, log=True),
        "clip_range": trial.suggest_float("clip_range", 0.1, 0.4),
        "gae_lambda": trial.suggest_float("gae_lambda", 0.8, 0.99),
    }

def opt_agent(trial):
    env = None
    try:
        env = make_vec_env()
        model = PPO(
            "CnnPolicy",
            env,
            verbose=0,
            device="mps",
            n_epochs=4,
            **opt_ppo(trial),
        )
        model.learn(total_timesteps=50000) # total trained steps
        mean_reward, std_reward = evaluate_policy(model, env,
            n_eval_episodes=5, # evaluate num of games
            deterministic=True,
        )
        print("Trial {}: mean={}, std={}".format(trial.number, mean_reward, std_reward))
        model.save(os.path.join(OPT_DIR, "trial_{}_best_model".format(trial.number)))
        return mean_reward
    except Exception as e:
        print("Trial {} FAILED: {}".format(trial.number, e))
        return -1000
    finally:
        if env:
            env.close()
def main():
    study = optuna.create_study(direction="maximize",)
    study.optimize(opt_agent, n_trials=500, n_jobs=1, timeout=9 * 3600)
    print("DONE")
    print("Best trial:", study.best_trial.number, "Best value:", study.best_value, "Best params:", study.best_params)

    
if __name__ == "__main__":
    main()


[I 2026-08-14 22:01:53,293] A new study created in memory with name: no-name-1798139b-6c08-4508-a4a0-25b12941adbf
[I 2026-08-14 22:04:54,340] Trial 0 finished with value: -307.0 and parameters: {'n_steps': 7488, 'gamma': 0.9950202247814492, 'learning_rate': 1.3649898308659877e-05, 'clip_range': 0.20356437319710907, 'gae_lambda': 0.8997394882179673}. Best is trial 0 with value: -307.0.


Trial 0: mean=-307.0, std=0.0
DONE
Best trial: 0 Best value: -307.0 Best params: {'n_steps': 7488, 'gamma': 0.9950202247814492, 'learning_rate': 1.3649898308659877e-05, 'clip_range': 0.20356437319710907, 'gae_lambda': 0.8997394882179673}
